# Laborator Cassandra & DataStax Astra DB
## Sistem de Gestionare Bibliotecă Online

**Tehnologii:** Apache Cassandra, DataStax Astra DB, Python, CQL

### Obiective:
- Înțelegerea modelării datelor în Cassandra
- Lucrul cu DataStax Astra DB
- Operații CRUD în CQL
- Query-uri specifice NoSQL
- Best practices pentru partition keys și clustering columns

## Configurare Inițială

### Instalare dependențe

In [ ]:
!pip install cassandra-driver
!pip install pandas

### Conectare la [DataStax Astra DB](https://dtsx.io/40kQpI6)

**Pași pentru configurare:**
1. Descarcă Secure Connect Bundle din Astra DB
2. Creează un Token de aplicație (cu rol Database Administrator)
3. Actualizează valorile de mai jos cu datele tale

In [ ]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
import pandas as pd
from datetime import datetime, timedelta
import uuid

# Configurare conexiune Astra DB
cloud_config = {
    'secure_connect_bundle': 'path/to/secure-connect-database.zip'
}

auth_provider = PlainTextAuthProvider(
    username='Client ID',
    password='Client Secret'
)

cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider)
session = cluster.connect()

print("✓ Conectat la Astra DB cu succes!")

## Partea 1: Crearea Bazei de Date

### 1.1 Crearea Keyspace-ului

In [ ]:
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS biblioteca
    WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}
""")

session.set_keyspace('biblioteca')
print("✓ Keyspace 'biblioteca' creat și activat")

### 1.2 Crearea Tabelelor

Vom crea 4 tabele pentru diferite query patterns:

In [ ]:
session.execute("""
    CREATE TABLE IF NOT EXISTS carti (
        isbn TEXT PRIMARY KEY,
        titlu TEXT,
        autor TEXT,
        an_publicare INT,
        editura TEXT,
        gen TEXT,
        numar_pagini INT,
        limba TEXT,
        pret DECIMAL,
        stoc INT
    )
""")
print("✓ Tabel 'carti' creat")

session.execute("""
    CREATE TABLE IF NOT EXISTS carti_by_gen (
        gen TEXT,
        an_publicare INT,
        isbn TEXT,
        titlu TEXT,
        autor TEXT,
        pret DECIMAL,
        PRIMARY KEY (gen, an_publicare, isbn)
    ) WITH CLUSTERING ORDER BY (an_publicare DESC, isbn ASC)
""")
print("✓ Tabel 'carti_by_gen' creat")

session.execute("""
    CREATE TABLE IF NOT EXISTS utilizatori (
        user_id UUID PRIMARY KEY,
        nume TEXT,
        email TEXT,
        data_inregistrare TIMESTAMP,
        telefon TEXT,
        adresa TEXT
    )
""")
print("✓ Tabel 'utilizatori' creat")

session.execute("""
    CREATE TABLE IF NOT EXISTS imprumuturi (
        user_id UUID,
        data_imprumut TIMESTAMP,
        imprumut_id UUID,
        isbn TEXT,
        titlu_carte TEXT,
        data_returnare_prevazuta TIMESTAMP,
        data_returnare_efectiva TIMESTAMP,
        status TEXT,
        PRIMARY KEY (user_id, data_imprumut, imprumut_id)
    ) WITH CLUSTERING ORDER BY (data_imprumut DESC)
""")
print("✓ Tabel 'imprumuturi' creat")

print("\n✓ Toate tabelele au fost create cu succes!")

### 1.3 Popularea Bazei de Date

In [ ]:
carti_data = [
    ('978-973-46-7890-1', 'Mara', 'Ioan Slavici', 1906, 'Humanitas', 'Roman', 320, 'RO', 45.50, 15),
    ('978-973-50-6543-2', 'Moromeții', 'Marin Preda', 1955, 'Cartex', 'Roman', 650, 'RO', 55.00, 8),
    ('978-606-33-5432-3', 'Python Programming', 'John Smith', 2022, 'TechBooks', 'IT', 450, 'EN', 120.00, 25),
    ('978-606-33-6789-4', 'Data Science Essentials', 'Maria Johnson', 2023, 'DataPub', 'IT', 380, 'EN', 95.00, 12),
    ('978-973-46-1234-5', 'Enigma Otiliei', 'George Călinescu', 1938, 'Polirom', 'Roman', 420, 'RO', 38.00, 20),
    ('978-606-40-9876-6', 'Machine Learning Basics', 'Alex Chen', 2023, 'AI Press', 'IT', 520, 'EN', 145.00, 7),
    ('978-973-46-5555-7', 'Ion', 'Liviu Rebreanu', 1920, 'Litera', 'Roman', 380, 'RO', 42.00, 18),
    ('978-606-33-7777-8', 'Web Development 2024', 'Sarah Williams', 2024, 'WebTech', 'IT', 490, 'EN', 110.00, 30),
    ('978-973-50-8888-9', 'Baltagul', 'Mihail Sadoveanu', 1930, 'Humanitas', 'Nuvelă', 180, 'RO', 28.00, 22),
    ('978-606-40-9999-0', 'Database Design', 'Robert Taylor', 2023, 'TechBooks', 'IT', 410, 'EN', 105.00, 14)
]

insert_carte = session.prepare("""
    INSERT INTO carti (isbn, titlu, autor, an_publicare, editura, gen, numar_pagini, limba, pret, stoc)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

insert_carte_gen = session.prepare("""
    INSERT INTO carti_by_gen (gen, an_publicare, isbn, titlu, autor, pret)
    VALUES (?, ?, ?, ?, ?, ?)
""")

for carte in carti_data:
    session.execute(insert_carte, carte)
    session.execute(insert_carte_gen, (carte[5], carte[2], carte[0], carte[1], carte[3], carte[8]))

print(f"✓ {len(carti_data)} cărți inserate")

user_ids = [uuid.uuid4() for _ in range(5)]
utilizatori_data = [
    (user_ids[0], 'Ion Popescu', 'ion.popescu@email.ro', datetime.now() - timedelta(days=365), '0721234567', 'Str. Libertății 15, București'),
    (user_ids[1], 'Maria Ionescu', 'maria.ionescu@email.ro', datetime.now() - timedelta(days=200), '0732345678', 'Bd. Unirii 45, Cluj-Napoca'),
    (user_ids[2], 'Andrei Georgescu', 'andrei.g@email.ro', datetime.now() - timedelta(days=150), '0743456789', 'Str. Mihai Viteazu 8, Iași'),
    (user_ids[3], 'Elena Dumitrescu', 'elena.d@email.ro', datetime.now() - timedelta(days=90), '0754567890', 'Aleea Rozelor 23, Timișoara'),
    (user_ids[4], 'Cristian Popa', 'cristian.popa@email.ro', datetime.now() - timedelta(days=30), '0765678901', 'Str. Primăverii 67, Constanța')
]

insert_user = session.prepare("""
    INSERT INTO utilizatori (user_id, nume, email, data_inregistrare, telefon, adresa)
    VALUES (?, ?, ?, ?, ?, ?)
""")

for user in utilizatori_data:
    session.execute(insert_user, user)

print(f"✓ {len(utilizatori_data)} utilizatori inserați")

imprumuturi_data = [
    (user_ids[0], datetime.now() - timedelta(days=10), uuid.uuid4(), '978-973-46-7890-1', 'Mara', datetime.now() + timedelta(days=4), None, 'activ'),
    (user_ids[0], datetime.now() - timedelta(days=25), uuid.uuid4(), '978-606-33-5432-3', 'Python Programming', datetime.now() - timedelta(days=11), datetime.now() - timedelta(days=8), 'returnat'),
    (user_ids[1], datetime.now() - timedelta(days=5), uuid.uuid4(), '978-973-50-6543-2', 'Moromeții', datetime.now() + timedelta(days=9), None, 'activ'),
    (user_ids[2], datetime.now() - timedelta(days=20), uuid.uuid4(), '978-606-33-6789-4', 'Data Science Essentials', datetime.now() - timedelta(days=6), datetime.now() - timedelta(days=3), 'returnat'),
    (user_ids[3], datetime.now() - timedelta(days=3), uuid.uuid4(), '978-606-40-9876-6', 'Machine Learning Basics', datetime.now() + timedelta(days=11), None, 'activ')
]

insert_imprumut = session.prepare("""
    INSERT INTO imprumuturi (user_id, data_imprumut, imprumut_id, isbn, titlu_carte, 
                            data_returnare_prevazuta, data_returnare_efectiva, status)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""")

for imprumut in imprumuturi_data:
    session.execute(insert_imprumut, imprumut)

print(f"✓ {len(imprumuturi_data)} împrumuturi inserate")
print("\n✓ Baza de date a fost populată cu succes!")

## Partea 2: Exerciții Rezolvate

### Exercițiul 1: Interogări Simple (SELECT)

In [ ]:
print("=== Toate cărțile ===")
rows = session.execute("SELECT * FROM carti")
for row in rows:
    print(f"{row.titlu} - {row.autor} ({row.an_publicare})")

print("\n" + "="*50 + "\n")

print("=== Căutare după ISBN ===")
isbn_cautat = '978-973-46-7890-1'
result = session.execute(
    "SELECT * FROM carti WHERE isbn = %s",
    (isbn_cautat,)
)
carte = result.one()
print(f"Titlu: {carte.titlu}")
print(f"Autor: {carte.autor}")
print(f"Preț: {carte.pret} RON")
print(f"Stoc: {carte.stoc} exemplare")

### Exercițiul 2: Filtrare și Sortare

In [ ]:
print("=== Cărți IT (cele mai recente) ===")
rows = session.execute(
    "SELECT * FROM carti_by_gen WHERE gen = 'IT'"
)
for row in rows:
    print(f"{row.an_publicare}: {row.titlu} - {row.autor} ({row.pret} RON)")

print("\n" + "="*50 + "\n")

print("=== Romane moderne ===")
rows = session.execute(
    "SELECT * FROM carti_by_gen WHERE gen = 'Roman' AND an_publicare >= 1924"
)
for row in rows:
    print(f"{row.titlu} ({row.an_publicare})")

### Exercițiul 3: Operații UPDATE

In [ ]:
print("=== Actualizare stoc ===")
isbn_update = '978-606-33-5432-3'

result = session.execute(
    "SELECT titlu, stoc FROM carti WHERE isbn = %s",
    (isbn_update,)
)
carte = result.one()
print(f"Înainte: {carte.titlu} - Stoc: {carte.stoc}")

session.execute(
    "UPDATE carti SET stoc = 30 WHERE isbn = %s",
    (isbn_update,)
)

result = session.execute(
    "SELECT stoc FROM carti WHERE isbn = %s",
    (isbn_update,)
)
print(f"După: Stoc: {result.one().stoc}")

print("\n" + "="*50 + "\n")

print("=== Actualizare preț ===")
session.execute(
    "UPDATE carti SET pret = 49.99 WHERE isbn = %s",
    ('978-973-46-7890-1',)
)
print("✓ Preț actualizat")

### Exercițiul 4: Lucrul cu Împrumuturi

In [ ]:
print("=== Împrumuturi utilizator ===")
user_id_exemplu = user_ids[0]

rows = session.execute(
    "SELECT * FROM imprumuturi WHERE user_id = %s",
    (user_id_exemplu,)
)

for row in rows:
    print(f"\nCarte: {row.titlu_carte}")
    print(f"Data împrumut: {row.data_imprumut.strftime('%d.%m.%Y')}")
    print(f"Status: {row.status}")
    if row.status == 'activ':
        zile_ramase = (row.data_returnare_prevazuta - datetime.now()).days
        print(f"Zile rămase: {zile_ramase}")

print("\n" + "="*50 + "\n")

print("=== Returnare carte ===")
rows = session.execute(
    "SELECT * FROM imprumuturi WHERE user_id = %s AND status = 'activ' LIMIT 1 ALLOW FILTERING",
    (user_id_exemplu,)
)
imprumut = rows.one()

session.execute("""
    UPDATE imprumuturi 
    SET status = 'returnat', data_returnare_efectiva = %s
    WHERE user_id = %s AND data_imprumut = %s AND imprumut_id = %s
""", (datetime.now(), imprumut.user_id, imprumut.data_imprumut, imprumut.imprumut_id))

print(f"✓ Cartea '{imprumut.titlu_carte}' a fost returnată")

### Exercițiul 5: Agregări și Statistici (Client-Side)

In [ ]:
print("=== Statistici bibliotecă ===")

rows = session.execute("SELECT stoc, pret FROM carti")
total_carti = sum(row.stoc for row in rows)
rows = session.execute("SELECT stoc, pret FROM carti")
valoare_stoc = sum(float(row.stoc * row.pret) for row in rows)

print(f"Total cărți în stoc: {total_carti}")
print(f"Valoare stoc: {valoare_stoc:.2f} RON")

rows = session.execute("SELECT gen FROM carti")
genuri = {}
for row in rows:
    genuri[row.gen] = genuri.get(row.gen, 0) + 1

print("\nCărți pe gen:")
for gen, count in genuri.items():
    print(f"  {gen}: {count}")

rows = session.execute("SELECT gen, pret FROM carti")
preturi_gen = {}
for row in rows:
    if row.gen not in preturi_gen:
        preturi_gen[row.gen] = []
    preturi_gen[row.gen].append(float(row.pret))

print("\nPreț mediu pe gen:")
for gen, preturi in preturi_gen.items():
    print(f"  {gen}: {sum(preturi)/len(preturi):.2f} RON")

## Partea 3: Exerciții Propuse (rezolvați 5 la alegere)

### Exercițiul 6: Query-uri de bază

In [ ]:
# TODO 6.1: Afișează toate cărțile în limba română
# Hint: Folosește tabelul 'carti' și filtrează după limba='RO'



In [ ]:
# TODO 6.2: Găsește toate cărțile mai scumpe de 100 RON
# ATENȚIE: Cassandra nu permite filtrare după coloane non-key fără ALLOW FILTERING
# Trebuie să facem filtrarea în Python după ce aducem toate datele



### Exercițiul 7: Inserare date noi

In [ ]:
# TODO 7.1: Adaugă 3 cărți noi în ambele tabele (carti și carti_by_gen)
# Alegeți voi datele pentru cărți



In [ ]:
# TODO 7.2: Adaugă un utilizator nou
# Generați un UUID nou pentru user_id



### Exercițiul 8: Gestionare împrumuturi

In [ ]:
# TODO 8.1: Creează un nou împrumut pentru un utilizator existent
# - Alegeți un user_id din lista user_ids
# - Alegeți o carte din baza de date
# - Data returnare prevăzută: 14 zile de la împrumut
# - Status: 'activ'



In [ ]:
# TODO 8.2: Afișează toate împrumuturile active din sistem
# Trebuie să iterați prin toți utilizatorii și să verificați împrumuturile lor



In [ ]:
# TODO 8.3: Găsește toate cărțile care au întârziere la returnare
# - Împrumuturile cu status='activ' și data_returnare_prevazuta < data curentă



### Exercițiul 9: Actualizări complexe

In [ ]:
# TODO 9.1: Actualizează prețurile tuturor cărților IT cu +10%
# Trebuie să:
# 1. Găsiți toate cărțile IT
# 2. Pentru fiecare carte, actualizați prețul în AMBELE tabele



In [ ]:
# TODO 9.2: Modifică stocul pentru cărțile cu stoc < 10
# Aduceți stocul la 20 de exemplare pentru toate cărțile cu stoc sub 10



### Exercițiul 10: Rapoarte și statistici

In [ ]:
# TODO 10.1: Creați un raport cu top 5 cele mai scumpe cărți
# Afișați: Titlu, Autor, Preț, Gen
# Sortați descrescător după preț



In [ ]:
# TODO 10.2: Calculați pentru fiecare utilizator:
# - Numărul total de împrumuturi
# - Numărul de împrumuturi active
# - Numărul de împrumuturi returnate



### Exercițiul 11: Query-uri avansate

In [ ]:
# TODO 11.1: Găsiți toate cărțile publicate între 2020-2024
# Folosiți tabelul carti_by_gen pentru genul 'IT'



In [ ]:
# TODO 11.2: Creați o funcție care verifică disponibilitatea unei cărți
# Funcția primește ISBN și returnează True dacă stoc > 0

def verifica_disponibilitate(isbn):
    # Scrieți codul aici:
    pass

# Testați funcția
print(verifica_disponibilitate('978-606-33-5432-3'))

In [ ]:
# TODO 11.3: Implementați o funcție de căutare cărți după autor
# Funcția primește numele autorului și returnează toate cărțile lui

def cauta_dupa_autor(autor):
    # Scrieți codul aici:
    pass

# Testați funcția
carti_gasite = cauta_dupa_autor('Ioan Slavici')
for carte in carti_gasite:
    print(carte)

### Exercițiul 12: Bonus - Denormalizare

In [ ]:
# TODO 12.1: Creați un nou tabel 'carti_by_autor' pentru a căuta rapid după autor
# Primary key: (autor, an_publicare, isbn)
# Clustering order: DESC pe an_publicare



In [ ]:
# TODO 12.2: Populați noul tabel cu datele existente din tabelul 'carti'



In [ ]:
# TODO 12.3: Testați noul tabel - găsiți toate cărțile lui 'Marin Preda'



### Exercițiul 13: Bonus - Batch Operations

In [ ]:
# TODO 13.1: Folosiți batch statements pentru a adăuga simultan:
# - 3 cărți noi
# - 2 utilizatori noi
# Hint: Folosiți cassandra.query.BatchStatement

from cassandra.query import BatchStatement



## Cleanup (Rulați la final)

In [ ]:
# Închidere conexiune
cluster.shutdown()
print("✓ Conexiune închisă")

## Resurse suplimentare
- [Apache Cassandra Docs](https://cassandra.apache.org/doc/latest/)
- [DataStax Astra DB Docs](https://docs.datastax.com/en/astra-db-serverless/index.html)
- [Cassandra Data Modeling](https://docs.datastax.com/en/cql/astra/data-modeling/methodology.html)
- Resurse video oficiale și semi-oficiale: 
	[1](https://academy.datastax.com/courses)
    [2](https://www.youtube.com/@DataStax/videos)
    [3](https://youtube.com/playlist?list=PLm-EPIkBI3YrmTV_kFeHH9MnOIjnK99nd&si=kdNykOTUY4SVVQ2x) 
- Soluții Cloud: 

    [DataStacks Astra DB](https://dtsx.io/40kQpI6)

    [Amazon Keyspaces for Apache Cassandra](https://aws.amazon.com/keyspaces/)

    [Azure Cosmos DB - Cassandra API](https://azure.microsoft.com/en-us/products/cosmos-db/)
    
    [Instaclustr (acum parte din NetApp)](https://www.instaclustr.com/products/managed-apache-cassandra/)